# ctdam test notebook

Fully self-contained python environment running in your webbrowser. **Nothing needs to be installed**, everything should run out of the box. There is only one test file "EMB394_126-01_CTD_0129.hex" (with corresponding .xmlcon and .bl) available inside the file system, but you can upload your own ones as well.

This will install the necessary python packages. Needs to be run and cannot be edited.

In [ ]:
import micropip
await micropip.install("ctdam", deps=False)
await micropip.install(["gsw_xarray", "scipy", "tqdm", "matplotlib", "odf.sbe", "xmltodict", "munch", "tomlkit"])

## Basics

In [ ]:
import ctdam

You can read all supported CTD data file types via the `parse` command. Only a file path is needed and optionally, the data can directly be cropped to the downcast.

In [ ]:
ds = ctdam.parse("EMB394_126-01_CTD_0129.hex", downcast_only=True)

The result is an [xarray Dataset](https://docs.xarray.dev/en/stable/user-guide/data-structures.html#dataset), which can be thought of as the 1:1 expression of a netCDF file inside python. A ton of functionality can accessed for this structure and throughout the data manipulation one is always just one simple `ds.to_netcdf()` away from saving the changes as netCDF to disk. These also render quite nicely when printed in jupyter notebooks.

In [ ]:
ds

You see that the CTD data and the corresponding metadata are well structured and can be accessed quite simply just by printing. Additionally, you can access all kinds of information via native xarray functionality.

In [ ]:
ds.pressure

In [ ]:
ds.temperature.attrs

In [ ]:
ds.attrs["path_to_source_file"]

You might have recognised the [CF](https://cfconventions.org/) naming convention being used as parameter description in `ctdam`. Together with the usage of netCDF as main data file structure, this shall make adhering to global oceanographic best practices as easy as possible.

On top of that, `ctdam` provides an xarray extension (called 'accessor'), to ease working with CTD data in this format. You can simply type in `ds.ctd.` and the Tab key to get an overview over the different functionality. 

In [ ]:
ds.ctd.

As you can see, there are different levels of organizing the additional functionality. But as before, you can select one of the options, for example `proc` (short for processing) and see whats available. This way you can explore whats directly possible on the xarray structure itself, without any additional imports from `ctdam`.

In [ ]:
ds.ctd.proc.

Whats not displayed though, are all the different processing modules that you can directly run ontop of your data. These can be displayed via `ds.proc.available_modules`. Note that I have ommited the `ctd` part here. This works for all accessor functionality of this library and is a little more convenient. But for clarity, you can also stick to the longer syntax. I will use the shorter one for the rest of this tutorial.

In [ ]:
ds.proc.available_modules

Lets bin the data as an example!

In [ ]:
binned_ds = ds.proc.bin
binned_ds

Now might be a good time to check out the arrays metadata a little more.

In [ ]:
binned_ds.proc.last

In [ ]:
binned_ds.meta.provenance

In [ ]:
binned_ds.access.binned

In [ ]:
binned_ds.access.bin_unit

In [ ]:
binned_ds.access.sample_rate

In [ ]:
binned_ds.access.size

In [ ]:
binned_ds.access.spans('temperature')

In [ ]:
binned_ds.meta.custom

For more complex processing workflows, you do not want to call all modules individually. `ctdam` does feature 'workflows' that you can configure and run streamlined on your data file. (Or on all of your files, via `ctdam.Casts`)

In [ ]:
ds.proc.workflow(['aircorrection', 'wildedit', 'wfilter', 'alignctd', 'celltm'])

In [ ]:
ds.meta.provenance

Lets have a look at our data!

In [ ]:
ds.vis.profile('salinity')

Bokeh plotting is not possible inside this environment, but you can check out [this prepared plot of the same data](https://dam-ctd-software.github.io/ctdam/plot.html). This can be created by `ds.vis.bokeh()`.

## Bottle handling

We can also incorporate bottle closing information into our CTD data. For that, we only need to call `ds.add.bottles` with a path to a Sea-Bird .bl file.

In [ ]:
import ctdam
ds = ctdam.parse("EMB394_126-01_CTD_0129.hex")
ds.add.bottles()
ds.bottle_info

You now have a new data array `bottle_info` that tracks to what kind of bottle a data point belong. You might also notice the 'global bottle ID' and the high values for the bottles. Thats a concept that was developed at the [IOW](https://www.iow.de/introduction-map.html) to apply unique identifiers to bottles throughout a cruise. You can display that bottle data as follows.

In [ ]:
btl = ds.access.btl_info
btl

In [ ]:
btl.access.pandas_dataframe

And if you, for some reason, need the original .btl file format, you can also export that.

In [ ]:
ds.export.to_btl()

`ctdam` can parse a lot of CTD data files, most notably all Sea-Bird files. You can therefore read in the newly created .btl file via its specific parser.

In [ ]:
btl_file = ctdam.parser.BottleFile("EMB394_126-01_CTD_0129.btl")

In [ ]:
btl_file.df

The great advantage of this workflow is the timing of the .btl file creation, as you can process the data and write the file afterwards to disk, something that is not possible with Sea-Birds routines.